In [140]:
import torch
import torch.nn.functional as F 
import matplotlib.pyplot as plt 

In [141]:
words = open("names.txt", "r").read().splitlines()
words[:3]

['emma', 'olivia', 'ava']

In [142]:
chars = sorted(list(set("".join(words))))

stoi = {s: i+1 for i, s in enumerate(chars)}
stoi["."] = 0 

itos = {i:s for s, i in stoi.items()}

In [143]:
# Build the dataset 
block_size = 3 
X, Y = [], [] 

for w in words:
   # print(w)

    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print("".join(itos[i] for i in context), "---->", itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [144]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [145]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [146]:
sum(p.nelement() for p in parameters) # total number of paramters 

3481

In [147]:
for p in parameters: 
    p.requires_grad = True 

In [ ]:
# Since we have figured out the range where we think lies the optimal learning rate, we create a range of values between those two points 


# lr = torch.linspace(0.001, 1, steps=1000)

# This is a better way of re-implementing the code above 
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

tensor([0.0010, 0.0020, 0.0030, 0.0040, 0.0050, 0.0060, 0.0070, 0.0080, 0.0090,
        0.0100, 0.0110, 0.0120, 0.0130, 0.0140, 0.0150, 0.0160, 0.0170, 0.0180,
        0.0190, 0.0200, 0.0210, 0.0220, 0.0230, 0.0240, 0.0250, 0.0260, 0.0270,
        0.0280, 0.0290, 0.0300, 0.0310, 0.0320, 0.0330, 0.0340, 0.0350, 0.0360,
        0.0370, 0.0380, 0.0390, 0.0400, 0.0410, 0.0420, 0.0430, 0.0440, 0.0450,
        0.0460, 0.0470, 0.0480, 0.0490, 0.0500, 0.0510, 0.0520, 0.0530, 0.0540,
        0.0550, 0.0560, 0.0570, 0.0580, 0.0590, 0.0600, 0.0610, 0.0620, 0.0630,
        0.0640, 0.0650, 0.0660, 0.0670, 0.0680, 0.0690, 0.0700, 0.0710, 0.0720,
        0.0730, 0.0740, 0.0750, 0.0760, 0.0770, 0.0780, 0.0790, 0.0800, 0.0810,
        0.0820, 0.0830, 0.0840, 0.0850, 0.0860, 0.0870, 0.0880, 0.0890, 0.0900,
        0.0910, 0.0920, 0.0930, 0.0940, 0.0950, 0.0960, 0.0970, 0.0980, 0.0990,
        0.1000, 0.1010, 0.1020, 0.1030, 0.1040, 0.1050, 0.1060, 0.1070, 0.1080,
        0.1090, 0.1100, 0.1110, 0.1120, 

In [151]:
for _ in range(100):

    # mini-batch construct 
    ix = torch.randint(0, X.shape[0], (32,))

    # forward pass 
    # C[x[ix]] means we only want 32 rows of X which we will the be used to be indexed in C
    emb = C[X[ix]] # (32, 3, 2)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) # (32, 100)
    logits = h @ W2 + b2 # (32, 27)


    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdims=True)
    # loss = -probs[torch.arange(32), Y].log().mean()

    # this one line of code is equivalent the the three commented lines above 
    # Y[ix] also select does same 32 rows corresponding Y label values 
    loss = F.cross_entropy(logits, Y[ix])
    print(loss.item())

    # backward pass 
    for p in parameters:
        p.grad = None

    loss.backward() 

    for p in parameters: 
        p.data -= 0.1 * p.grad 


3.9344301223754883
4.44337797164917
2.7906601428985596
3.324108839035034
3.252162456512451
4.548473834991455
4.022263050079346
4.50246000289917
3.4471912384033203
3.1190638542175293
5.090511322021484
4.032238006591797
3.2812724113464355
3.5050039291381836
3.209510326385498
3.9547667503356934
3.1425228118896484
3.1032795906066895
3.465482234954834
3.5617318153381348
3.1351263523101807
3.2436819076538086
3.0716021060943604
3.123340129852295
3.2740445137023926
3.7942142486572266
3.445451259613037
3.2270257472991943
3.292386293411255
3.2055881023406982
3.516068458557129
3.105839967727661
3.3962907791137695
2.941054105758667
3.288146734237671
3.267568826675415
3.76383376121521
3.4890501499176025
3.4468960762023926
3.960968255996704
3.4200756549835205
3.468771457672119
3.5205769538879395
3.371727228164673
3.2941370010375977
3.4670305252075195
3.046780824661255
3.089186191558838
2.5939841270446777
3.301215887069702
2.9132628440856934
2.8768692016601562
2.8810343742370605
2.545754909515381
3.5

In [149]:
# generate mini-batches 
# Select 32 training examples from X which will be 32 indexes 
torch.randint(0, X.shape[0], (32,))

tensor([153269,  32345, 166987, 171203,  71039, 209958, 161610,  40307,  74291,
         97334,  52263, 208644,  37551, 212050,  96533,  82957,  92691, 206628,
        169667, 173604, 122153, 160188, 135244,  65515,  42191, 144598, 196005,
         51581, 188801, 184191,  74071, 106565])